## What are the greeks?

Options aren't just as simple as a set price. They have properties describing how they change depending on the underlying market. We are focusing back on European options specifically.

$\Delta$ (Delta) - rate of change of option price w.r.t. underlying price. For a call, Delta is form 0 to 1 and for a put its from -1 to 1.
$$\Delta_{\text{call}} = N(d_1), \qquad \Delta_{\text{put}} = N(d_1) - 1$$

$\Delta$ can be interptered as the hedge ratio, so an option with $\Delta$ = -0.5 hedges for half of one share of the underlying. An ATM option should have $|\Delta|$ = 0.5


$\Gamma$ (Gamma) - rate of change of Delta w.r.t. underlying price.
$$\Gamma = \frac{n(d_1)}{S_0 \, \sigma \sqrt{T}}$$

The equation is the same for calls and puts. $\Gamma$ is highest for or close to ATM options and increases closer to the expirey date. When using options to hedge, a high $\Gamma$ means you will have to adjust your hedge often.


$\nu$ (Vega) - sensitivity of option price to change in implied volitility.
$$\nu = S_0 \, n(d_1) \sqrt{T}$$

The equation is identical for calls and puts and is always positive as higher volitility makes options more valuable. $\nu$ is highest for or close to ATM options.


$\Theta$ (Theta) - Rate of change of option price w.r.t. time (days). 
$$\Theta_{\text{call}} = -\frac{S_0 \, n(d_1) \sigma}{2\sqrt{T}} - r K e^{-rT} N(d_2)$$
$$\Theta_{\text{put}} = -\frac{S_0 \, n(d_1) \sigma}{2\sqrt{T}} + r K e^{-rT} N(-d_2)$$

$\Theta$ is generally negative as the price of the underlying has less time to change before expirey. Large values of $\Gamma$ comes with large values of $\Theta$ too.

$\rho$ (Rho) - sesnitivity of option prcie to changes in the risk-free rate.
$$\rho_{\text{call}} = K T e^{-rT} N(d_2)$$
$$\rho_{\text{put}} = -K T e^{-rT} N(-d_2)$$

$\rho$ is positive for calls and negative for puts. It is mostly considered for long-term options.


## Estimating the greeks

We are going to estimate the values of the greeks using finite difference methods, this is done by changing one of the inputs by a small amount and observing the result. Where sensible, we use central difference (using +h and -h) as it is more accurate, but for $\Theta$, we use forward difference as time only moves in one direction.
$$\Delta \approx \frac{C(S_0 + h) - C(S_0 - h)}{2h}, \qquad h = 1.0$$

$$\Gamma \approx \frac{C(S_0 + h) - 2C(S_0) + C(S_0 - h)}{h^2}, \qquad h = 1.0$$

$$\nu \approx \frac{C(\sigma + h) - C(\sigma - h)}{2h}, \qquad h = 0.01$$

$$\rho \approx \frac{C(r + h) - C(r - h)}{2h}, \qquad h = 0.01$$

$$\Theta \approx \frac{C(T - \Delta t) - C(T)}{\Delta t}, \qquad \Delta t = \frac{1}{252}$$

We predict that the values obtianed through this method will be close to the closed form formulae as layed out earlier. This method will then be useful later when estimating the greeks for a Heston model where we do not have a closed form solution.

In [24]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from utils import black_scholes

Parameters

In [25]:
# from 04
S0    = 100    # initial price
K = 105 # strike price
sigma = 0.2272 # gbm volitility
r = 0.05 # risk free rate
T     = 1.0    # time horizon
N     = int(252 * T)    # time steps (number of trading days)

# new parameters
h_S     = 1.0         # change for Delta and Gamma
h_sigma = 0.01        # change for Vega
h_r     = 0.01        # change for Rho
dt      = 1 / 252     # one trading day, for Theta

Black-Scholes method

In [26]:
def bs_d1d2(S0, K, r, sigma, T):
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return d1, d2

In [27]:
d1, d2 = bs_d1d2(S0, K, r, sigma, T)

delta_call = stats.norm.cdf(d1)
delta_put  = stats.norm.cdf(d1) - 1

gamma = stats.norm.pdf(d1) / (S0 * sigma * np.sqrt(T))

vega = S0 * stats.norm.pdf(d1) * np.sqrt(T) / 100 # divided by 100 to see per 1% change in volitility

theta_call = (-(S0 * stats.norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) # divided by 252 to get per trading day
              - r * K * np.exp(-r * T) * stats.norm.cdf(d2)) / 252
theta_put  = (-(S0 * stats.norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
              + r * K * np.exp(-r * T) * stats.norm.cdf(-d2)) / 252

rho_call = K * T * np.exp(-r * T) * stats.norm.cdf(d2)  / 100 # divided by 100 to get ber 1% rate change
rho_put  = -K * T * np.exp(-r * T) * stats.norm.cdf(-d2) / 100


print(f"{'Greek':<10} {'Call':>15} {'Put':>15}")
print(f"{'Delta':<10} {delta_call:>15.6f} {delta_put:>15.6f}")
print(f"{'Gamma':<10} {gamma:>15.6f} {gamma:>15.6f}")
print(f"{'Vega':<10} {vega:>15.6f} {vega:>15.6f}")
print(f"{'Theta':<10} {theta_call:>15.6f} {theta_put:>15.6f}")
print(f"{'Rho':<10} {rho_call:>15.6f} {rho_put:>15.6f}")

Greek                 Call             Put
Delta             0.547333       -0.452667
Gamma             0.017435        0.017435
Vega              0.396131        0.396131
Theta            -0.026912       -0.007094
Rho               0.456336       -0.542455


Finite difference method

In [28]:

delta_fd_call = (black_scholes(S0 + h_S, K, r, sigma, T, 'call') -
                 black_scholes(S0 - h_S, K, r, sigma, T, 'call')) / (2 * h_S)
delta_fd_put  = (black_scholes(S0 + h_S, K, r, sigma, T, 'put') -
                 black_scholes(S0 - h_S, K, r, sigma, T, 'put')) / (2 * h_S)


gamma_fd_call = (black_scholes(S0 + h_S, K, r, sigma, T, 'call') -
                 2 * black_scholes(S0, K, r, sigma, T, 'call') +
                 black_scholes(S0 - h_S, K, r, sigma, T, 'call')) / (h_S ** 2)
gamma_fd_put  = (black_scholes(S0 + h_S, K, r, sigma, T, 'put') -
                 2 * black_scholes(S0, K, r, sigma, T, 'put') +
                 black_scholes(S0 - h_S, K, r, sigma, T, 'put')) / (h_S ** 2)


vega_fd_call = (black_scholes(S0, K, r, sigma + h_sigma, T, 'call') -
                black_scholes(S0, K, r, sigma - h_sigma, T, 'call')) / (2 * h_sigma * 100)
vega_fd_put  = (black_scholes(S0, K, r, sigma + h_sigma, T, 'put') -
                black_scholes(S0, K, r, sigma - h_sigma, T, 'put')) / (2 * h_sigma * 100)


theta_fd_call = (black_scholes(S0, K, r, sigma, T - dt, 'call') -
                 black_scholes(S0, K, r, sigma, T, 'call')) / dt / 252
theta_fd_put  = (black_scholes(S0, K, r, sigma, T - dt, 'put') -
                 black_scholes(S0, K, r, sigma, T, 'put')) / dt / 252


rho_fd_call = (black_scholes(S0, K, r + h_r, sigma, T, 'call') -
               black_scholes(S0, K, r - h_r, sigma, T, 'call')) / (2 * h_r * 100)
rho_fd_put  = (black_scholes(S0, K, r + h_r, sigma, T, 'put') -
               black_scholes(S0, K, r - h_r, sigma, T, 'put')) / (2 * h_r * 100)


print(f"{'Greek':<10} {'Call':>15} {'Put':>15}")
print(f"{'Delta':<10} {delta_fd_call:>15.6f} {delta_fd_put:>15.6f}")
print(f"{'Gamma':<10} {gamma_fd_call:>15.6f} {gamma_fd_put:>15.6f}")
print(f"{'Vega':<10} {vega_fd_call:>15.6f} {vega_fd_put:>15.6f}")
print(f"{'Theta':<10} {theta_fd_call:>15.6f} {theta_fd_put:>15.6f}")
print(f"{'Rho':<10} {rho_fd_call:>15.6f} {rho_fd_put:>15.6f}")

Greek                 Call             Put
Delta             0.547288       -0.452712
Gamma             0.017433        0.017433
Vega              0.396129        0.396129
Theta            -0.026929       -0.007110
Rho               0.456300       -0.542508


Compare

In [29]:

greeks      = ['Delta', 'Gamma', 'Vega', 'Theta', 'Rho']

analytical_call = [delta_call, gamma,    vega,    theta_call, rho_call]
analytical_put  = [delta_put,  gamma,    vega,    theta_put,  rho_put]

fd_call     = [delta_fd_call, gamma_fd_call, vega_fd_call, theta_fd_call, rho_fd_call]
fd_put      = [delta_fd_put,  gamma_fd_put,  vega_fd_put,  theta_fd_put,  rho_fd_put]

print("Calls")
print(f"{'Greek':<10} {'Analytical':>15} {'Finite Dif':>15} {'Error':>15} {'Error (%)':>10}")
for g, a, f in zip(greeks, analytical_call, fd_call):
    print(f"{g:<10} {a:>15.6f} {f:>15.6f} {abs(a-f):>15.2e} {abs(100*(a-f)/a):>10.2e}")

print("\nPuts")
print(f"{'Greek':<10} {'Analytical':>15} {'Finite Dif':>15} {'Error':>15} {'Error (%)':>10}")
for g, a, f in zip(greeks, analytical_put, fd_put):
    print(f"{g:<10} {a:>15.6f} {f:>15.6f} {abs(a-f):>15.2e} {abs(100*(a-f)/a):>10.2e}")

Calls
Greek           Analytical      Finite Dif           Error  Error (%)
Delta             0.547333        0.547288        4.43e-05   8.08e-03
Gamma             0.017435        0.017433        2.26e-06   1.29e-02
Vega              0.396131        0.396129        1.64e-06   4.14e-04
Theta            -0.026912       -0.026929        1.73e-05   6.43e-02
Rho               0.456336        0.456300        3.67e-05   8.03e-03

Puts
Greek           Analytical      Finite Dif           Error  Error (%)
Delta            -0.452667       -0.452712        4.43e-05   9.78e-03
Gamma             0.017435        0.017433        2.26e-06   1.29e-02
Vega              0.396131        0.396129        1.64e-06   4.14e-04
Theta            -0.007094       -0.007110        1.53e-05   2.16e-01
Rho              -0.542455       -0.542508        5.33e-05   9.83e-03


## Evaluation

The errors for the finite difference approximations are extremely small. Theta shows a slightly higher error as it was not calculated using the central difference method.